In [9]:
from pyspark.sql.functions import col, datediff, current_date, to_date, when

# --- a) Silver Students ---
df_students = spark.read.table("bronze_students")
silver_students = df_students.withColumn("registration_date", to_date(col("registration_date"))) \
    .withColumn("account_age_days", datediff(current_date(), col("registration_date"))) \
    .withColumn("age", col("age").cast("integer")) \
    .withColumn("total_courses_enrolled", col("total_courses_enrolled").cast("integer"))

# Added overwriteSchema to fix the merge field error
silver_students.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_students")

# --- b) Silver Assessments ---
df_assessments = spark.read.table("bronze_assessments")
silver_assessments = df_assessments \
    .withColumn("enrollment_date", to_date(col("enrollment_date"))) \
    .withColumn("completion_date", to_date(col("completion_date"))) \
    .withColumn("score", col("score").cast("decimal(5,2)")) \
    .withColumn("time_spent_hours", col("time_spent_hours").cast("decimal(10,2)")) \
    .withColumn("completion_days", datediff(col("completion_date"), col("enrollment_date"))) \
    .withColumn("pass_flag", when(col("score") >= 60, True).otherwise(False))

# Added overwriteSchema to fix the merge field error
silver_assessments.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_assessments")

# --- c) Silver Courses ---
df_courses = spark.read.table("bronze_courses")
silver_courses = df_courses \
    .withColumn("price_usd", col("price_usd").cast("decimal(10,2)")) \
    .withColumn("enrollment_count", col("enrollment_count").cast("integer")) \
    .withColumn("revenue_per_enrollment", col("price_usd") * col("enrollment_count"))

# Added overwriteSchema to fix the merge field error
silver_courses.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_courses")

# --- Output Verification ---
print("-" * 40)
print(f"silver_students row count:    {silver_students.count()}")
print(f"silver_assessments row count: {silver_assessments.count()}")
print(f"silver_courses row count:     {silver_courses.count()}")
print("-" * 40)
print("Silver Layer processing complete with Schema Overwrite.")

StatementMeta(, 5258c985-eb05-4956-b073-8f2938585da9, 11, Finished, Available, Finished, True)

----------------------------------------
silver_students row count:    500
silver_assessments row count: 4000
silver_courses row count:     100
----------------------------------------
Silver Layer processing complete with Schema Overwrite.


In [10]:
# Define paths for the Silver Files folder
silver_files_path = "Files/silver"

# Save the DataFrames physically to the Files/silver folder
silver_students.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_files_path}/students_delta")
silver_assessments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_files_path}/assessments_delta")
silver_courses.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_files_path}/courses_delta")

print("Silver Delta files now saved in Files/silver directory.")

StatementMeta(, 5258c985-eb05-4956-b073-8f2938585da9, 12, Finished, Available, Finished, True)

Silver Delta files now saved in Files/silver directory.
